In [6]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import torchvision.models as models
from torchsummary import summary

In [10]:
class DepthwiseConv(nn.Module):
    def __init__(self, in_channels, kernel_size, stride, padding, bias):
        super(DepthwiseConv, self).__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels,
            bias=bias
        )
        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU6(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class PointwiseConv(nn.Module):
    def __init__(self, in_channels, out_channels, bias):
        super(PointwiseConv, self).__init__()
        self.pointwise = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=bias
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU6(inplace=True)

    def forward(self, x):
        x = self.pointwise(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super(DepthwiseSeparableConv, self).__init__()
        self.depthwise = DepthwiseConv(
            in_channels=in_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.pointwise = PointwiseConv(
            in_channels=in_channels,
            out_channels=out_channels,
            bias=False
        )

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

class MobileNet(nn.Module):
    def __init__(self, num_classes=1000, alpha=1.0):
        super(MobileNet, self).__init__()

        self.config = [
            [3, 32, 2],
            [32, 64, 1],
            [64, 128, 2],
            [128, 128, 1],
            [128, 256, 2],
            [256, 256, 1],
            [256, 512, 2],
            [512, 512, 1],
            [512, 512, 1],
            [512, 512, 1],
            [512, 512, 1],
            [512, 512, 1],
            [512, 1024, 2],
            [1024, 1024, 1],
        ]

        # Apply width multiplier
        new_config = []
        for i, (in_c, out_c, stride) in enumerate(self.config):
            if i == 0:
                new_in_c = in_c
                new_out_c = 32
            else:
                new_in_c = int(in_c * alpha)
                new_out_c = int(out_c * alpha)
            new_config.append([new_in_c, new_out_c, stride])
        self.config = new_config

        self.initial_conv = nn.Sequential(
            nn.Conv2d(3, self.config[0][1], kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.config[0][1]),
            nn.ReLU6(inplace=True)
        )

        # Depthwise separable convolutions
        self.layers = nn.ModuleList()
        for in_c, out_c, stride in self.config[1:]:
            self.layers.append(
                DepthwiseSeparableConv(in_c, out_c, stride)
            )

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(self.config[-1][1], num_classes)
        self._initialize_weights()

    def forward(self, x):
        x = self.initial_conv(x)

        for layer in self.layers:
            x = layer(x)

        x = self.avg_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Usage
model = MobileNet(num_classes=1000, alpha=1.0)
total_params = count_parameters(model)
print(f"Total parameters: {total_params:,}")

Total parameters: 4,231,976


In [11]:
model = models.mobilenet_v2(weights="DEFAULT")
total_params = count_parameters(model)
print(f"Total parameters: {total_params:,}")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 130MB/s]

Total parameters: 3,504,872
